# Auditando as partidas do fornecedor

## Demanda da aula 2

Com o ambiente no ar e os arquivos do fornecedor lidos, chegou o primeiro pedido do time de produto: estatísticas de
cada Copa e de cada seleção para usar nas transmissões. Durante um jogo, o narrador quer poder dizer coisas como
"esta é a Copa com a maior média de gols" ou "esta seleção tem o melhor saldo de gols em Copas".

Antes de calcular qualquer número, vem a pergunta que todo engenheiro de dados faz: **posso confiar nesse dado?** No
fim da aula 1, uma olhada rápida já mostrou um problema: partidas sem o identificador (`MatchID`). E o público chegou
como texto, por causa da vírgula de milhar. Se uma olhada rápida achou isso, é hora de conferir o arquivo com calma:
qualquer erro no dado vai parar na transmissão.

Nesta aula, você audita e limpa as partidas, corrige os tipos, calcula as estatísticas e junta as informações dos
três arquivos do fornecedor.

| Etapa | O que fazer | Notebook |
|---|---|---|
| Task 1 | Auditar as partidas: quantas chegaram, quantas estão vazias ou repetidas e quantas sobram | `01_transformacao_vs_acao` |
| Task 2 | Padronizar os nomes das colunas e corrigir os tipos do público e da data | `02_select_filter_withcolumn` |
| Task 3 | Calcular as estatísticas por Copa e por seleção | `03_agregacoes` |
| Task 4 | Juntar escalações, sede e campeão às partidas e levar o código para o projeto | `04_join` |

**Pronto quando:** as estatísticas por Copa e por seleção chegam ao mesmo total de gols, nenhuma conversão de tipo
gerou nulos sem a gente saber, e a limpeza das partidas está num arquivo do projeto, `src/wc_pipeline/transformacoes.py`.

## Task 1: auditar as partidas

Quantas partidas chegaram, quantas estão vazias ou repetidas e quantas sobram.

A partir desta aula, a célula da sessão já vem pronta: o código do vídeo 1.4, com outro `appName`.

In [ ]:
# a sessão Spark deste notebook: já vem pronta (é o código do vídeo 1.4, com outro appName)
import sys
sys.path.insert(0, "../../src")

from pyspark.sql import SparkSession, functions as F

spark = (
    SparkSession.builder
    .appName("arena-auditoria-partidas")
    .master("local[*]")
    .config("spark.sql.warehouse.dir", "../../data/warehouse")
    .getOrCreate()
)

RAW = "../../data/raw"

spark

## Transformações e ações

| O método devolve... | Tipo | O que acontece | Exemplos |
|---|---|---|---|
| outro **DataFrame** | transformação | só acrescenta um passo ao plano | `select`, `filter`, `withColumn`, `groupBy().agg()`, `join`, `orderBy`, `dropDuplicates` |
| um **valor Python**, ou grava um arquivo | ação | o Spark executa o plano | `show`, `count`, `first`, `take`, `collect`, `write` |

**Lazy evaluation**: as transformações só são calculadas quando uma ação pede um resultado. Até lá, o Spark só anota
os passos, como uma receita.

`goleadas` é um DataFrame: só uma receita. Nenhum job novo na Spark UI.

## As ações

**`show(n)`**: as primeiras `n` linhas, formatadas.

**`count()`**: conta as linhas; percorre o dado inteiro.

**`collect()`**: traz **todas** as linhas para a memória do driver. Use só quando o resultado for pequeno, por exemplo
depois de agregar.

## A auditoria das partidas

1. **Total bruto:** linhas no arquivo.
2. **Linhas vazias:** sem `MatchID`.
3. **Duplicatas:** o mesmo `MatchID` mais de uma vez.
4. **Total limpo:** o que sobra sem vazias e sem duplicatas.

As duplicatas são a mesma partida repetida?

Linhas idênticas: a mesma partida enviada duas vezes. Regra de limpeza: **tirar as linhas sem `MatchID` e manter uma
linha por `MatchID`**.

## Resultado da task 1

576 linhas no arquivo, 3 vazias e 11 repetidas: sobram **562 partidas**. Próxima task: nomes e tipos das colunas.

## Para consultar depois

- **`first()` e `take(n)`** trazem 1 ou `n` linhas para o driver, como objetos `Row`. São baratas numa leitura com
  filtros simples, mas, depois de uma agregação ou ordenação, o Spark precisa calcular tudo antes de devolver a primeira
  linha.
- **Cada ação executa o plano de novo.** O Spark não guarda o resultado de uma ação para a próxima: dois `count()`
  seguidos leem o arquivo duas vezes. Guardar um resultado para reaproveitar é o `cache()`, da aula 4.
- **`printSchema()` não é ação.** O schema já é conhecido quando o plano é montado, então nenhuma linha é lida.
- **`toPandas()`** tem o mesmo cuidado do `collect()`: traz tudo para a memória do driver.

## Encerrando

Ao terminar o notebook, encerre a sessão Spark. Isso libera a memória e a porta da Spark UI para o próximo notebook.

In [ ]:
spark.stop()